# 00 — Hardware Benchmark & Auto-Detect Protocol
**Customer360 Navigator Enterprise Suite — Sprint 1 Kickoff Notebook**

## Purpose
Implements Master Execution Plan Section 17.2 (Auto-detect & benchmark-first protocol). Detects this
machine's real hardware, benchmarks CPU thread counts (4/8/12/16) and sequential-vs-parallel 5-fold
StratifiedKFold CV on a SYNTHETIC dataset (never real Customer360 data), and writes
`configs/hardware_benchmark_summary.json` as the single source of truth every later BP notebook reads for
its thread count, CV mode, and CPU/GPU/NPU setting.

## Standing rules this notebook follows
- **Execution boundary** (Master Execution Plan Section 12.2): Claude wrote this notebook; it does not run
  it. You run it on your own machine, and the real, live-measured numbers below become the project's
  benchmark record. Nothing here was pre-computed or carried over from a prior session.
- **Zero-fabrication** (Section 12.1): every number below is computed live during this run. The benchmark
  dataset is explicitly synthetic and labeled as such — never confused with real CFPB/BANKING77 data.
- **WARP** (Section 15/17): never targets 100% CPU/RAM; keeps a 92% RAM / 95% CPU-thread ceiling as a
  safety cap, not a floor to pad toward.
- **Idempotent**: re-running this notebook overwrites its output files in place — safe to re-run any time
  hardware, drivers, or installed packages change.
- **PROJECT_STRUCTURE_LOCKED.md rule #3**: project root is resolved via an environment-variable override
  first, then a bounded upward walk — never a hardcoded absolute path. No absolute local path is printed
  below (this repo may be shared publicly later via `github_repo/`).

## Outputs (both written, idempotent overwrite-in-place)
- `configs/hardware_benchmark_summary.json` — the single source of truth every later BP notebook reads
- `notebooks/00_hardware_benchmark/artifacts/hardware_benchmark_summary.json` — local copy

## Prerequisites
Run `pip install -r requirements.txt` from the project root first. `torch` and `onnxruntime` are optional —
their absence only narrows GPU/NPU detection, it does not fail this notebook.

## If a structural check below fails
It raises `AssertionError` with the failing check named. Do not edit the check to make it pass — fix the
real cause (see `LESSONS_LEARNED_APPLIED.md` for the failure classes this notebook already guards against).


In [ ]:

"""
Customer360 Navigator Enterprise Suite - 00_hardware_benchmark.ipynb
WARP Section 17.2. Single consolidated code cell (platform convention, matches every
AMEX RiskIQ / Home Credit RiskIQ notebook). Idempotent - safe to re-run.
"""

import os, sys, json, time, platform, shutil, warnings, builtins, functools
from pathlib import Path
from datetime import datetime, timezone

warnings.filterwarnings("ignore")

# Force every print() in this cell to flush immediately. Real symptom reported on a live run:
# "no outputs are printed" after a full re-run, with no error either - the most likely cause is
# output buffering in the front-end/execution mode being used (varies by Jupyter client, and is
# worse when a cell runs long, as SECTION 3's parallel benchmark below can). This does not fix a
# genuine hang, but it guarantees every checkpoint below appears the instant it runs rather than
# only after the whole cell finishes, so a hang becomes visible as "stopped after line X" instead
# of a totally silent cell.
print = functools.partial(builtins.print, flush=True)

# ============================================================
# SECTION 1: Project root resolution (PROJECT_STRUCTURE_LOCKED.md rule #3)
# Never hardcode an absolute path - env var override, then an upward walk, then a bounded
# downward search (see LESSONS_LEARNED_APPLIED.md - real bug: an upward-only walk fails
# whenever the kernel's cwd is a PARENT of the project folder rather than inside it, e.g.
# VS Code's Jupyter extension defaulting the kernel cwd to the workspace root instead of the
# notebook's own folder).
# ============================================================
def _find_project_root() -> Path:
    marker = "PROJECT_STRUCTURE_LOCKED.md"
    env_override = os.environ.get("C360_PROJECT_ROOT")
    if env_override:
        if (Path(env_override) / marker).exists():
            return Path(env_override)
        raise RuntimeError(
            f"C360_PROJECT_ROOT is set to {env_override!r} but {marker} was not found there. "
            "Fix the environment variable rather than removing this check."
        )

    start = Path.cwd()

    # Upward walk: handles the common case where the kernel's cwd is inside the project tree.
    cur = start
    for _ in range(8):
        if (cur / marker).exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent

    # Bounded downward search (depth <= 3, skips hidden dirs): handles the case where the
    # kernel's cwd is a PARENT of the project folder - the upward walk above cannot find a
    # marker that lives in a child directory. Depth-limited to stay fast even under a large
    # Documents folder containing other projects.
    for depth_root, dirnames, filenames in os.walk(start):
        rel_depth = len(Path(depth_root).relative_to(start).parts)
        if rel_depth > 3:
            dirnames[:] = []
            continue
        dirnames[:] = [d for d in dirnames if not d.startswith(".")]
        if marker in filenames:
            return Path(depth_root)

    raise RuntimeError(
        f"Could not resolve PROJECT_ROOT: no {marker} found by walking up from {start}, nor by "
        "searching up to 3 levels below it. This usually means the notebook kernel's working "
        "directory is neither inside nor the direct parent of the project folder (a common "
        "cause: VS Code's Jupyter extension defaults the kernel's cwd to the workspace root "
        "instead of the notebook's own folder). Fix: add a cell at the TOP of this notebook "
        "(before this cell runs) with:\n"
        '    import os; os.environ["C360_PROJECT_ROOT"] = r"C:\\Users\\rnand\\Documents\\'
        'Customer360_Navigator_Enterprise_Suite"\n'
        "then re-run from the top."
    )

PROJECT_ROOT = _find_project_root()
CONFIGS_DIR = PROJECT_ROOT / "configs"
ARTIFACTS_DIR = PROJECT_ROOT / "notebooks" / "00_hardware_benchmark" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"[OK] Project root resolved: {(PROJECT_ROOT / 'PROJECT_STRUCTURE_LOCKED.md').exists()}")
# (Absolute path deliberately not printed - see markdown intro, privacy convention.)

# ============================================================
# SECTION 2: Hardware detection (live, this run only - zero-fabrication)
# ============================================================
import psutil

def detect_gpu():
    result = {"method": None, "gpu_name": None, "cuda_available_via_torch": None, "notes": []}
    if platform.system() == "Windows":
        try:
            import subprocess
            out = subprocess.run(
                ["wmic", "path", "win32_VideoController", "get", "name"],
                capture_output=True, text=True, timeout=10,
            )
            lines = [ln.strip() for ln in out.stdout.splitlines() if ln.strip() and "Name" not in ln]
            if lines:
                result["method"] = "wmic"
                result["gpu_name"] = lines
        except Exception as e:
            result["notes"].append(f"wmic GPU detection failed: {e}")
    try:
        import torch
        result["cuda_available_via_torch"] = torch.cuda.is_available()
    except ImportError:
        result["notes"].append("torch not installed - CUDA availability not checked")
    result["notes"].append(
        "Master Execution Plan Section 17.1/17.6: no discrete/NVIDIA GPU expected on this hardware "
        "(integrated AMD Radeon 8xx, no CUDA path). Informational only - GPU is not used as a training "
        "accelerator by this notebook."
    )
    return result

def detect_npu():
    result = {"onnxruntime_providers": None, "npu_provider_detected": None, "notes": []}
    try:
        import onnxruntime as ort
        providers = ort.get_available_providers()
        result["onnxruntime_providers"] = providers
        result["npu_provider_detected"] = any(
            key in p for p in providers for key in ("NPU", "VitisAI", "DML")
        )
    except ImportError:
        result["notes"].append("onnxruntime not installed - NPU execution-provider availability not checked")
    result["notes"].append(
        "Master Execution Plan Section 17.6: the NPU has no code path for XGBoost/LightGBM/CatBoost/"
        "scikit-learn/spaCy. Only relevant to a future ONNX-exported model - not benchmarked here."
    )
    return result

def detect_library_versions():
    libs = ["polars", "duckdb", "numpy", "pandas", "sklearn", "xgboost", "catboost",
            "lightgbm", "spacy", "sentence_transformers", "transformers", "numba", "shap"]
    versions = {}
    for lib in libs:
        try:
            mod = __import__(lib)
            versions[lib] = getattr(mod, "__version__", "unknown")
        except ImportError:
            versions[lib] = "NOT INSTALLED"
    return versions

def detect_hardware():
    info = {"timestamp_utc": datetime.now(timezone.utc).isoformat()}
    info["os"] = platform.system()
    info["os_release"] = platform.release()
    info["python_version"] = sys.version.split()[0]
    info["cpu_physical_cores"] = psutil.cpu_count(logical=False)
    info["cpu_logical_threads"] = psutil.cpu_count(logical=True)
    try:
        freq = psutil.cpu_freq()
        info["cpu_freq_current_mhz"] = round(freq.current, 1) if freq else None
        info["cpu_freq_max_mhz"] = round(freq.max, 1) if freq and freq.max else None
    except Exception as e:
        info["cpu_freq_current_mhz"] = None
        info["cpu_freq_max_mhz"] = None
        info["cpu_freq_error"] = str(e)
    # Known real limitation (not a bug in this notebook): on Windows, psutil.cpu_freq() reads
    # Win32_Processor via WMI, which exposes only the CPU's nominal/base clock speed under both
    # CurrentClockSpeed and MaxClockSpeed - it does not expose live turbo-boost frequency, which
    # Windows does not reliably surface through this API at all. So current==max here (e.g. both
    # reading the base clock) is EXPECTED on Windows and does not mean the CPU is stuck at that
    # speed, and it is NOT the same as the CPU's actual rated min/max turbo range (a spec value
    # this notebook cannot verify live and therefore never fabricates). This has no effect on the
    # thread-count recommendation below, which is based on logical core count, not clock speed.
    info["cpu_freq_windows_wmi_limitation"] = (
        info["os"] == "Windows"
        and info["cpu_freq_current_mhz"] is not None
        and info["cpu_freq_current_mhz"] == info["cpu_freq_max_mhz"]
    )
    vm = psutil.virtual_memory()
    info["ram_total_gb"] = round(vm.total / (1024 ** 3), 2)
    info["ram_available_gb"] = round(vm.available / (1024 ** 3), 2)
    try:
        du = shutil.disk_usage(str(PROJECT_ROOT))
        info["disk_free_gb_at_project_root"] = round(du.free / (1024 ** 3), 2)
        info["disk_total_gb_at_project_root"] = round(du.total / (1024 ** 3), 2)
    except Exception as e:
        info["disk_free_gb_at_project_root"] = None
        info["disk_error"] = str(e)
    try:
        temps = psutil.sensors_temperatures()
        info["cpu_temperature_available"] = bool(temps)
    except Exception:
        info["cpu_temperature_available"] = False
    info["gpu_detection"] = detect_gpu()
    info["npu_detection"] = detect_npu()
    info["library_versions"] = detect_library_versions()
    return info

HW = detect_hardware()
print(f"[OK] CPU: {HW['cpu_physical_cores']} physical / {HW['cpu_logical_threads']} logical threads")
print(f"[OK] RAM: {HW['ram_total_gb']} GB total / {HW['ram_available_gb']} GB available")
print(f"[OK] Disk free at project root: {HW['disk_free_gb_at_project_root']} GB")
print(f"[OK] CPU freq current/max (MHz): {HW['cpu_freq_current_mhz']} / {HW['cpu_freq_max_mhz']}")
if HW.get("cpu_freq_windows_wmi_limitation"):
    print("[LIMITATION] current == max above is expected on Windows: psutil reads this from WMI's "
          "Win32_Processor, which exposes only the nominal/base clock speed for both fields - it does "
          "not expose live turbo-boost frequency. This is NOT the CPU's actual rated min/max turbo "
          "range (check your CPU's spec sheet or Task Manager > Performance for that) and it does not "
          "affect the thread-count recommendation below, which uses core/thread count, not clock speed.")
print(f"[INFO] CPU temperature sensors available: {HW['cpu_temperature_available']}")
print(f"[INFO] GPU detected: {HW['gpu_detection'].get('gpu_name')}")
print(f"[INFO] CUDA available via torch: {HW['gpu_detection'].get('cuda_available_via_torch')}")
print(f"[INFO] onnxruntime providers: {HW['npu_detection'].get('onnxruntime_providers')}")
print(f"[INFO] Library versions: {HW['library_versions']}")

# ============================================================
# SECTION 3: CPU thread-count + sequential-vs-parallel CV benchmark
# SYNTHETIC data only - never real Customer360 data. Keeps only the configuration with
# the lowest stable wall-clock time, never the highest utilization percentage.
# ============================================================
import joblib
from sklearn.datasets import make_classification
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression

N_SAMPLES_BENCHMARK = 50_000
N_FEATURES_BENCHMARK = 40

print(f"\n[SYNTHETIC] Generating a synthetic benchmark dataset (NOT real Customer360 data) - "
      f"{N_SAMPLES_BENCHMARK} rows x {N_FEATURES_BENCHMARK} features, for hardware timing only.")
X_bench, y_bench = make_classification(
    n_samples=N_SAMPLES_BENCHMARK,
    n_features=N_FEATURES_BENCHMARK,
    n_informative=20,
    n_redundant=10,
    weights=[0.92, 0.08],  # mimics an imbalanced BP3-style escalation target
    random_state=42,
)

LOGICAL_THREADS = HW["cpu_logical_threads"] or 16
THREAD_COUNTS = [t for t in [4, 8, 12, 16] if t <= LOGICAL_THREADS] or [LOGICAL_THREADS]
cv_folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

thread_benchmark_results = []
for n_jobs in THREAD_COUNTS:
    model = LogisticRegression(max_iter=200, n_jobs=(n_jobs if n_jobs > 1 else None))

    print(f"[BENCH] starting n_jobs={n_jobs}: sequential CV (n_jobs=1)...")
    t0 = time.perf_counter()
    scores_seq = cross_val_score(model, X_bench, y_bench, cv=cv_folds, scoring="roc_auc", n_jobs=1)
    elapsed_sequential = time.perf_counter() - t0
    print(f"[BENCH] n_jobs={n_jobs}: sequential CV done in {elapsed_sequential:.2f}s.")

    # Real bug hit on a live Windows/Jupyter run: joblib's default process-based "loky" backend
    # uses Windows' spawn start method, which re-imports the entry-point module in every child
    # process. Inside an interactive Jupyter kernel (no __main__ guard, no freezable script) this
    # can hang silently with zero output and no error - see LESSONS_LEARNED_APPLIED.md #12. Forcing
    # a thread-based backend here avoids process spawn entirely for this benchmark step; threads
    # cannot show the wall-clock parallel speedup a process backend would on CPU-bound sklearn work,
    # but a correct, non-hanging measurement is more useful than a faster one that may never return.
    print(f"[BENCH] starting n_jobs={n_jobs}: parallel CV (n_jobs={min(5, n_jobs)}, threading backend)...")
    t0 = time.perf_counter()
    with joblib.parallel_backend("threading", n_jobs=min(5, n_jobs)):
        scores_par = cross_val_score(model, X_bench, y_bench, cv=cv_folds, scoring="roc_auc", n_jobs=min(5, n_jobs))
    elapsed_parallel = time.perf_counter() - t0
    print(f"[BENCH] n_jobs={n_jobs}: parallel CV done in {elapsed_parallel:.2f}s.")

    thread_benchmark_results.append({
        "n_jobs_model": n_jobs,
        "sequential_cv_seconds": round(elapsed_sequential, 3),
        "parallel_cv_seconds": round(elapsed_parallel, 3),
        "mean_roc_auc_sequential": round(float(scores_seq.mean()), 4),
        "mean_roc_auc_parallel": round(float(scores_par.mean()), 4),
        "ram_percent_at_measurement": psutil.virtual_memory().percent,
    })
    print(f"[BENCH] n_jobs={n_jobs}: sequential CV={elapsed_sequential:.2f}s, "
          f"parallel CV={elapsed_parallel:.2f}s, RAM={psutil.virtual_memory().percent:.1f}%")

nested_flagged = [r for r in thread_benchmark_results if r["n_jobs_model"] > 1]
print(f"\n[CHECK] Configurations combining parallel CV folds with a multi-threaded model "
      f"(potential nested parallelism): {len(nested_flagged)} of {len(thread_benchmark_results)} - "
      f"the recommendation below picks the single fastest STABLE config measured, not the most parallel one.")

best_config = min(thread_benchmark_results, key=lambda r: r["parallel_cv_seconds"])
print(f"[RESULT] Fastest stable configuration measured this run: n_jobs={best_config['n_jobs_model']}, "
      f"{best_config['parallel_cv_seconds']}s parallel CV.")

# ============================================================
# SECTION 4: Resource ceilings (WARP) + write outputs (idempotent overwrite-in-place)
# ============================================================
RAM_CEILING_FRACTION = 0.92
CPU_CEILING_FRACTION = 0.95

recommended = {
    "recommended_n_jobs": best_config["n_jobs_model"],
    "recommended_cv_mode": (
        "parallel" if best_config["parallel_cv_seconds"] < best_config["sequential_cv_seconds"] else "sequential"
    ),
    "ram_ceiling_gb": round(HW["ram_total_gb"] * RAM_CEILING_FRACTION, 2),
    "cpu_thread_ceiling": max(1, int(LOGICAL_THREADS * CPU_CEILING_FRACTION)),
    "gpu_recommended": False,
    "gpu_reason": "No CUDA path detected on this hardware; not benchmarked as a training accelerator (Section 17.6).",
    "npu_recommended": False,
    "npu_reason": "No NPU code path exists for the classical 6-model benchmark set (Section 17.6).",
}

summary = {
    "notebook": "00_hardware_benchmark.ipynb",
    "generated_by": "user real run (execution-boundary rule - Claude never executes this itself)",
    "hardware_detected": HW,
    "thread_benchmark_results": thread_benchmark_results,
    "recommended_configuration": recommended,
    "warning": "SYNTHETIC benchmark data only - these are TIMING measurements, not any Customer360 result.",
}

summary_path = CONFIGS_DIR / "hardware_benchmark_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

artifact_copy_path = ARTIFACTS_DIR / "hardware_benchmark_summary.json"
with open(artifact_copy_path, "w") as f:
    json.dump(summary, f, indent=2)

print("\n[SAVED] Hardware benchmark summary written (idempotent overwrite) to configs/ and this "
      "notebook's artifacts/ folder.")
print(f"[RECOMMENDATION] n_jobs={recommended['recommended_n_jobs']}, "
      f"cv_mode={recommended['recommended_cv_mode']}, "
      f"RAM ceiling={recommended['ram_ceiling_gb']} GB, "
      f"CPU thread ceiling={recommended['cpu_thread_ceiling']} of {LOGICAL_THREADS}")
print("[NEXT] Every later BP notebook must read configs/hardware_benchmark_summary.json for its thread "
      "count / CV mode / GPU-NPU setting - never hardcode a value again (Master Execution Plan Section 17.2).")

# Inline comparison table (never rely on the saved file alone - platform convention)
import pandas as pd
from IPython.display import display
report_df = pd.DataFrame(thread_benchmark_results)
display(report_df)

# ============================================================
# SECTION 5: Structural integrity checks (always PASS if the notebook completes cleanly,
# raise AssertionError otherwise - never edited to force a pass)
# ============================================================
checks = [
    ("cpu_logical_threads_detected", bool(HW["cpu_logical_threads"])),
    ("ram_total_detected", bool(HW["ram_total_gb"])),
    ("at_least_one_thread_benchmark_completed", len(thread_benchmark_results) > 0),
    ("summary_json_written", summary_path.exists()),
    ("artifact_copy_written", artifact_copy_path.exists()),
    ("no_nested_parallelism_in_recommendation", recommended["recommended_n_jobs"] <= LOGICAL_THREADS),
]

print("\n=== INTEGRITY CHECKS ===")
all_passed = True
for name, passed in checks:
    status = "PASS" if passed else "FAIL"
    all_passed = all_passed and passed
    print(f"[{status}] {name}")

if not all_passed:
    raise AssertionError("One or more structural integrity checks FAILED - see output above. "
                          "Fix the real cause; do not edit this check to force a pass.")

print("\n[ALL CHECKS PASSED] 00_hardware_benchmark.ipynb complete. "
      "Proceed to Sprint 1's data acquisition/profiling step next.")
